In [1]:
# --- Robust CSV reader for Windows-encoded CSVs ---


def read_csv_safely(path, **kwargs):
    """
    Tries common encodings for Windows/Excel-exported CSVs.
    Returns a DataFrame or raises the last error if all encodings fail.
    """
    # Common encodings for Windows CSVs and Excel 'Save As'
    tried = []
    for enc in ["utf-8", "utf-8-sig", "windows-1252", "latin-1", "utf-16", "utf-16le", "utf-16be"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            tried.append((enc, str(e)[:120]))
            continue
    # As a last resort, try replacing bad bytes
    try:
        return pd.read_csv(path, encoding="windows-1252", encoding_errors="replace", **kwargs)
    except Exception as e:
        tried.append(("windows-1252+replace", str(e)[:120]))
        raise ValueError("Could not read CSV with common encodings. Tried:\n" +
                         "\n".join([f"- {enc}: {err}" for enc, err in tried]))



In [14]:

import re

def clean_illegal_chars(df):
    # Remove non-printable illegal Excel characters
    illegal_pattern = r"[\x00-\x1F]"
    return df.applymap(lambda x: re.sub(illegal_pattern, "", str(x)) if isinstance(x, str) else x)

# Clean both DataFrames before# Clean both DataFrames before saving
#new_sku_df = clean_illegal_chars(new_sku_df)


import pandas as pd
import decimal


def sci_to_plain_str(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    try:
        d = decimal.Decimal(s)
        # If integer-like, format without exponent or decimals
        if d == d.to_integral_value():
            return format(d.quantize(decimal.Decimal(1)), 'f')
        else:
            return format(d.normalize(), 'f')
    except Exception:
        return s  # leave as-is if not convertible

#old_df['UPC'] = old_df['UPC'].apply(sci_to_plain_str).astype('string')




In [19]:
import pandas as pd
import os

# Change directory to Downloads folder
downloads_path = os.path.expanduser(r"C:\Users\muhammad.farooqui\Downloads\Suppliers")
os.chdir(downloads_path)

OldFile = "10-11-26 ZyroFisher.csv"  # <-- rename if needed
NewFile = "21-01-26 ZyroFisher.csv"  # <-- rename if needed


old_df = read_csv_safely(OldFile)# Read the Excel file into a DataFrame with a function
new_df = read_csv_safely(NewFile) # Read the Excel file into a DataFrame with a function


print(old_df['UPC'].head(10))
print(new_df['UPC'].dtypes)


#Selecting Columns to drop
cols_to_drop = [
    "VatCode",
    "StockIndicator",
    "StockDueIn",
    "LongDescription",
    "ImageUrl",
    "Currency",
    "BriefDescription",
    "OrangePrice",
    "SilverPrice",
    "BronzePrice"
]

# Drop only if the column exists (avoids errors)
old_df = old_df.drop(columns=[c for c in cols_to_drop if c in old_df.columns])
new_df = new_df.drop(columns=[c for c in cols_to_drop if c in new_df.columns])



#old_df['UPC'] = pd.to_numeric(old_df['UPC'].astype('int64'), errors='coerce')

#old_df['UPC'] = old_df['UPC'].apply(sci_to_int).astype('Int64')  # pandas nullable int
old_df['UPC'] = old_df['UPC'].apply(sci_to_plain_str).astype('string')


pd.set_option('display.max_columns', None)     # show all columns
pd.set_option('display.width', 5000)           # very wide line so columns don't wrap
pd.set_option('display.max_colwidth', None)    # show full column contents


print("OLD shape:", old_df.shape)
print("NEW shape:", new_df.shape)


#print(new_df.head(10))  # Display the first few rows of the dataframe
#print(old_df.head(10))


0    7.10846E+11
1    7.10846E+11
2    7.10846E+11
3    7.10846E+11
4    7.10846E+11
5    7.10846E+11
6    7.10846E+11
7    7.10846E+11
8    7.10846E+11
9    7.10846E+11
Name: UPC, dtype: object
object
OLD shape: (15821, 11)
NEW shape: (15599, 11)


In [ ]:
import pandas as pd
import decimal

# 1) Read UPC as string to avoid auto-conversion to float/scientific notation
df = pd.read_csv("input.csv", dtype={"UPC": "string"})

# 2) Convert scientific-notation strings to integers precisely
def to_int_decimal(x):
    if pd.isna(x):
        return pd.NA
    try:
        d = decimal.Decimal(str(x))  # handles '7.10846E+11' exactly
        return int(d.to_integral_value(rounding=decimal.ROUND_HALF_UP))
    except Exception:
        return pd.NA

df["UPC"] = df["UPC"].apply(to_int_decimal).astype("Int64")  # pandas nullable int

# (Optional) Save
df.to_csv("output.csv", index=False)

In [52]:
#Preparing output DataFrames.
new_sku_df = pd.DataFrame()       # SKUs not found in OLD file
updating_sku_df = pd.DataFrame()  # SKUs found in both but content changed


In [53]:
#Loop through NEW file row by row

for idx, new_row in new_df.iterrows():
    
    sku = new_row["SKU"]
    
    # 1️⃣ Check if SKU exists in old file
    matching_old = old_df[old_df["SKU"] == sku]
    
    if matching_old.empty:
        # 2️⃣ SKU does not exist in old file → NEW SKU
        new_sku_df = pd.concat([new_sku_df, new_row.to_frame().T], ignore_index=True)
        continue  # move to next SKU
    
    # 3️⃣ SKU exists → compare content
    old_row = matching_old.iloc[0]  # take the first matching row
    
    # Compare entire row except SKU column
    cols_to_compare = [col for col in new_df.columns if col != "SKU" and col in old_df.columns]
    
    changed = False
    for col in cols_to_compare:
        if pd.isna(new_row[col]) and pd.isna(old_row[col]):
            continue  # both NaN, treat as equal
        if str(new_row[col]).strip() != str(old_row[col]).strip():
            changed = True
            break
    
    if changed:
        updating_sku_df = pd.concat([updating_sku_df, new_row.to_frame().T], ignore_index=True)


In [54]:
#Save result

new_sku_df.to_excel("New_SKUs.xlsx", index=False)
updating_sku_df.to_excel("Updating_SKUs.xlsx", index=False)

print("Done!")
print(f"New SKUs found: {len(new_sku_df)}")
print(f"SKUs needing update: {len(updating_sku_df)}")


Done!
New SKUs found: 1110
SKUs needing update: 14405


In [26]:

# 🔑 Define your unique key column(s) here:
KEY_COLS = ["SKU"]  # <-- e.g., ["SKU"] or ["SupplierID"] or ["Product Code"]

# Safety checks
for k in KEY_COLS:
    if k not in of.columns or k not in nf.columns:
        raise KeyError(f"Key '{k}' not found in both files. Available columns:\nOLD: {of.columns}\nNEW: {nf.columns}")

# Drop perfect-duplicates on keys + values (optional cleanliness)
of = of.drop_duplicates(subset=KEY_COLS + [c for c in of.columns if c not in KEY_COLS])
nf = nf.drop_duplicates(subset=KEY_COLS + [c for c in nf.columns if c not in KEY_COLS])

# Align to the same column set for comparison (union, then order)
all_cols = sorted(set(of.columns).union(set(nf.columns)))
of_aligned = of.reindex(columns=all_cols)
nf_aligned = nf.reindex(columns=all_cols)

In [29]:

# Set index to key
old_k = of_aligned.set_index(KEY_COLS, drop=False)
new_k = nf_aligned.set_index(KEY_COLS, drop=False)

# Same keys
common_keys = old_k.index.intersection(new_k.index)

# Columns to compare (excluding the key)
value_cols = [c for c in all_cols if c not in KEY_COLS]

# Detect changes
old_common = old_k.loc[common_keys, value_cols].sort_index()
new_common = new_k.loc[common_keys, value_cols].sort_index()

diff_mask = old_common != new_common
changed_keys = diff_mask.any(axis=1)

#Extract changed rows from the NEW file
changed_rows = new_k.loc[changed_keys.index]

In [ ]:
# Deleting unnecessary columns.
cols_to_delete = ["Date Updated", "Weight (g.)", "Pack", "MX", "Trade", "Qty", "Ltd Stock"]
df.drop(columns=cols_to_delete, inplace=True)
